# 02 · From judgments to policy

**You will:** compose focused judgments, separate independent and dependent questions, and choose a review threshold without using evaluation labels.

The model supplies judgments. Your Python policy controls routing. Detecting a refund request does **not** authorize a refund. Authentication, eligibility and payments remain ordinary business systems.

In [ ]:
# Find the course folder and load API settings without printing secret keys.
from pathlib import Path
import sys
import json
from IPython.display import display
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'course_utils').is_dir():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open Jupyter inside the course folder.')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from course_utils.config import load_settings, require_key
settings = load_settings(ROOT)
print('Python:', sys.version.split()[0], '| Mode:', settings.mode)
print('Jev key configured:', bool(settings.typesafe_api_key))
print('OpenAI key configured:', bool(settings.openai_api_key))

In [ ]:
# Load the course tickets; reference labels are for evaluation only.
tickets = []
for line in (ROOT / 'data/tickets.jsonl').read_text().splitlines():
    tickets.append(json.loads(line))

In [ ]:
# Define the three questions used for each ticket.
TEAMS = {
    'billing': 'Existing payments, refunds, invoices, and subscriptions.',
    'technical': 'Malfunctioning application features or service errors.',
    'account': 'Login, credentials, profiles, and membership.',
    'other': 'Sales, suggestions, general questions, or insufficient evidence.',
}
FRUSTRATION = [
    'Neutral or positive.',
    'Mild complaint or annoyance.',
    'Clear or repeated frustration.',
    'Intense anger or explicit outrage.',
]
TEAM_PROMPT = 'Which team should handle the ticket? For mixed intents choose the first actionable issue.'
REFUND_PROMPT = 'Does the customer explicitly request money back? Policy questions, cancellation alone, and negated requests are not requests.'
from typesafe_sdk import TypeSafeClient, Choice, Score, Noul, RetryPolicy
questions = {
    'team': Choice(instructions=TEAM_PROMPT, criteria=TEAMS),
    'frustration': Score(
        instructions='How frustrated is the customer, based on expressed tone rather than urgency?',
        criteria=FRUSTRATION,
    ),
    'refund_requested': Noul(instructions=REFUND_PROMPT),
}

In [ ]:
# Import the decision types and configure live calls or authentic replay.
from contextlib import nullcontext
from typesafe_sdk import TypeSafeClient, Choice, Noul, RetryPolicy
from course_utils.records import run_case
from course_utils.evaluation import route_team

RUN_DIR = ROOT / 'work/runs'
can_jev = settings.mode == 'recorded' or bool(settings.typesafe_api_key)
can_openai = settings.mode == 'recorded' or bool(settings.openai_api_key)


def jev_client():
    if settings.mode == 'recorded':
        return nullcontext(None)
    return TypeSafeClient(
        api_key=require_key(settings, 'typesafe'),
        retry=RetryPolicy(max_retries=1),
        timeout=30,
    )

In [ ]:
# Send one decision request and show its input, status, and answers.
def ask_jev(example_id, state, questions):
    print('State analysed:')
    display(state)
    if not can_jev:
        print('Skipped: configure TYPESAFE_API_KEY or select authentic recordings.')
        return None

    config = {
        'state': state,
        'questions': {name: question.model_dump(mode='json') for name, question in questions.items()},
        'max_retries': 1,
        'timeout': 30,
    }
    with jev_client() as client:
        def request():
            return client.system_one(
                model=settings.jev_new, state=state, questions=questions,
            )

        result = run_case(
            request, provider='typesafe', endpoint='systemone',
            model=settings.jev_new, example_id=example_id, config=config,
            question_names=list(questions), mode=settings.mode, directory=RUN_DIR,
        )

    print('Status:', result.status, '| Error:', result.error)
    display(result.answers)
    return result

## A Python policy uses the answers
Missing or uncertain team answers go to review. A refund request does not authorize a refund.

In [ ]:
# Turn model answers into a team, a review flag, and a priority.
def triage(answers, threshold):
    team_answer = answers.get('team', {})
    destination = 'review'
    if threshold is not None:
        destination = route_team(
            team_answer.get('choice', 'review'),
            team_answer.get('confidence', float('nan')),
            threshold,
        )
    frustration = answers.get('frustration', {}).get('score', 0)
    return {
        'team': destination,
        'review': destination == 'review',
        'priority': 'high' if frustration >= 2 else 'normal',
    }

## First tune on development examples

The three questions are independent and can be sent together. We try five team-confidence thresholds on the 12 development cases. Our teaching objective is at least 90% accuracy among automatically routed cases, maximizing coverage. If none qualifies, review everything. This tiny sample does not establish a safe production threshold.

A case with an API failure stays in the denominator and goes to review. Inspect raw records when answers are missing. Mixed intent can still be a semantic failure even when the selected first team matches our reference.

In [ ]:
# Keep development examples separate from held-out evaluation examples.
from course_utils.evaluation import choose_threshold, selective_metrics, classification_metrics

development_tickets = [ticket for ticket in tickets if ticket['split'] == 'dev']
evaluation_tickets = [ticket for ticket in tickets if ticket['split'] == 'eval']
development_results = []
if can_jev:
    for ticket in development_tickets:
        result = ask_jev(ticket['id'], {'ticket': ticket['text']}, questions)
        development_results.append(result)

In [ ]:
# Choose the review threshold using only development answers and labels.
development_correct = []
development_confidence = []
chosen_threshold = None
for ticket, result in zip(development_tickets, development_results):
    answer = result.answers.get('team', {})
    development_correct.append(answer.get('choice') == ticket['team'])
    development_confidence.append(answer.get('confidence', float('-inf')))

if development_results:
    chosen_threshold = choose_threshold(development_correct, development_confidence)
    print('Frozen threshold:', chosen_threshold if chosen_threshold is not None else 'review all')

In [ ]:
# Show how the review rate changes at different confidence thresholds.
threshold_rows = []
if development_results:
    for threshold in (0.5, 0.6, 0.7, 0.8, 0.9):
        accepted = [confidence >= threshold for confidence in development_confidence]
        metrics = selective_metrics(development_correct, accepted)
        threshold_rows.append({'threshold': threshold, **metrics})
    display(pd.DataFrame(threshold_rows))

## Evaluate once, with the frozen policy

Accuracy over all cases and accuracy among accepted cases answer different questions. Report both alongside review rate. A system that reviews every case cannot claim perfect automated accuracy.

In [ ]:
# Evaluate held-out tickets without changing the chosen threshold.
evaluation_results = []
if can_jev:
    for ticket in evaluation_tickets:
        result = ask_jev(ticket['id'], {'ticket': ticket['text']}, questions)
        evaluation_results.append(result)

In [ ]:
# Build a readable table containing the evidence, model answer, and policy decision.
evaluation_rows = []
for ticket, result in zip(evaluation_tickets, evaluation_results):
    decision = triage(result.answers, chosen_threshold)
    evaluation_rows.append({
        'id': ticket['id'],
        'state': ticket['text'],
        'expected': ticket['team'],
        'predicted': result.answers.get('team', {}).get('choice'),
        'status': result.status,
        **decision,
    })
if evaluation_rows:
    display(pd.DataFrame(evaluation_rows))

In [ ]:
# Report overall accuracy separately from accuracy among automatically routed cases.
if evaluation_rows:
    references = {row['id']: row['expected'] for row in evaluation_rows}
    predictions = {row['id']: row['predicted'] for row in evaluation_rows}
    correct = [row['predicted'] == row['expected'] for row in evaluation_rows]
    accepted = [not row['review'] for row in evaluation_rows]
    display(classification_metrics(references, predictions))
    display(selective_metrics(correct, accepted))
    print('This small teaching set does not establish general reliability.')

## Precision versus recall: choose a threshold for the use case

The earlier threshold used **team confidence** to decide whether to route automatically or request human review. Here we threshold the **Noul probability of a refund request** to classify a ticket as positive or negative. These are different policies.

`flag_as_refund_request = refund_probability >= threshold`

| Term | Meaning in this example |
|---|---|
| True positive (TP) | Correctly flagged a real refund request |
| False positive (FP) | Flagged a ticket that is not a refund request |
| False negative (FN) | Missed a real refund request |
| True negative (TN) | Correctly left a non-request unflagged |
| Precision = TP / (TP + FP) | Of the tickets we flagged, how many really ask for a refund? |
| Recall = TP / (TP + FN) | Of the real requests, how many did we catch? |

Lowering a threshold flags more tickets and cannot reduce recall on the same scored examples. Raising it flags fewer and often improves precision, but precision need not increase at every step.

We reuse the refund probabilities already obtained above: **this section makes no extra API calls**.

In [ ]:
# Pair development labels with their refund probabilities using ticket IDs.
from course_utils.evaluation import binary_threshold_metrics, select_binary_threshold

development_by_id = {result.example_id: result for result in development_results}
refund_labels = []
refund_probabilities = []
for ticket in development_tickets:
    refund_labels.append(ticket['refund_requested'])
    result = development_by_id.get(ticket['id'])
    answer = result.answers.get('refund_requested', {}) if result is not None else {}
    refund_probabilities.append(answer.get('noul'))

In [ ]:
# Compare false alarms, missed requests, precision, and recall at several thresholds.
refund_thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
refund_threshold_rows = []
if development_results:
    for threshold in refund_thresholds:
        metrics = binary_threshold_metrics(refund_labels, refund_probabilities, threshold)
        refund_threshold_rows.append(metrics)
    display(pd.DataFrame(refund_threshold_rows))
else:
    print('Run the development requests first; no precision/recall results are available yet.')

Metrics are computed on valid answers. The table also reports **coverage**, **missing answers**, and **unscored positives**, so API failures cannot disappear unnoticed. Missing answers go to human review, never to the negative class. This example refuses to select a threshold until the development set is fully scored.

Precision is undefined when nothing is flagged; recall is undefined when there are no scored positive examples. Neither case is displayed as perfect performance.

In [ ]:
# Plot both metrics against the probability threshold using development results only.
import matplotlib.pyplot as plt

if refund_threshold_rows:
    threshold_table = pd.DataFrame(refund_threshold_rows)
    figure, axes = plt.subplots(figsize=(7, 3))
    axes.plot(threshold_table['threshold'], threshold_table['precision'], 'o-', label='Precision')
    axes.plot(threshold_table['threshold'], threshold_table['recall'], 'o-', label='Recall')
    axes.set(xlabel='Refund-request probability threshold', ylabel='Metric', ylim=(0, 1.05))
    axes.legend()
    axes.grid(alpha=0.3)
    plt.show()

### Two use cases, two policies

- **Find requests for a human review queue:** Missing a real request is costly, while a human can dismiss a false alarm. Require at least **90% recall**, then choose the qualifying threshold with the best precision.
- **Trigger an expensive specialist workflow:** False alarms waste scarce capacity. Require at least **90% precision**, then choose the qualifying threshold with the best recall.

These are illustrative targets. Neither policy authorizes a refund or performs a real escalation. If no tested threshold meets the target, do not force a choice: use human review, improve the question, or collect better development data. Both use cases may select the same threshold on a small or easy dataset.

In [ ]:
# Select and freeze both policies using only the development threshold table.
refund_policies = {
    'Find requests for human review': select_binary_threshold(
        refund_threshold_rows, prioritize='recall', target=0.9,
    ),
    'Expensive specialist workflow': select_binary_threshold(
        refund_threshold_rows, prioritize='precision', target=0.9,
    ),
}
for policy_name, threshold in refund_policies.items():
    if threshold is None:
        print(policy_name, ': no qualifying policy; use human review.')
    else:
        print(policy_name, ': frozen threshold =', threshold)

### Check held-out performance once

The next cell reports the frozen thresholds on evaluation tickets. It does not search for a better threshold there. A development target can fail on held-out data; report that result rather than retuning until it passes. The tiny course dataset illustrates the method, not a reliable production guarantee.

In [ ]:
# Evaluate the frozen refund policies on held-out labels and existing responses.
evaluation_by_id = {result.example_id: result for result in evaluation_results}
evaluation_refund_labels = []
evaluation_refund_probabilities = []
for ticket in evaluation_tickets:
    evaluation_refund_labels.append(ticket['refund_requested'])
    result = evaluation_by_id.get(ticket['id'])
    answer = result.answers.get('refund_requested', {}) if result is not None else {}
    evaluation_refund_probabilities.append(answer.get('noul'))

refund_policy_report = []
if evaluation_results:
    for policy_name, threshold in refund_policies.items():
        if threshold is None:
            continue
        metrics = binary_threshold_metrics(
            evaluation_refund_labels, evaluation_refund_probabilities, threshold,
        )
        refund_policy_report.append({'policy': policy_name, **metrics})
    display(pd.DataFrame(refund_policy_report))
else:
    print('Run the held-out requests before evaluating the frozen policies.')

### Exercise

Raise the recall target to 100% for the review-queue use case, or raise the precision target to 95% for the expensive workflow. Predict which errors you are trying to reduce before inspecting the development results. This is an exercise on development data; do not use the held-out report to repeatedly tune the targets.

**Interpretation:** Higher recall aims to reduce missed requests; higher precision aims to reduce the fraction of false alarms among flagged requests. Inspect the TP/FP/FN counts, missing answers, and human workload—not just a single percentage. Keep the policy in review mode if the available evidence does not support your requirement.

## When a second call is needed

Questions in one request cannot branch on each other's answers. Here the first routing decision chooses whether to ask a billing-specific follow-up. This example does not send a message or perform a refund.

In [ ]:
# Ask a billing-specific follow-up only after the first answer selects billing.
if development_results:
    first_answer = development_results[0].answers.get('team', {})
    if first_answer.get('choice') == 'billing':
        followup_questions = {
            'duplicate': Noul(
                instructions='Does the customer report the same payment being taken more than once?',
            ),
        }
        followup_state = {'ticket': development_tickets[0]['text']}
        followup_result = ask_jev('duplicate-followup', followup_state, followup_questions)

## Combine several scores into one priority

A ticket can be severe, frustrating, and detailed to different degrees. Ask three independent Score questions in one request, then combine their answers with an explicit Python policy.

The rubrics have different lengths: severity and frustration range from **0 to 2**, while report quality ranges from **0 to 3**. Divide each score by its highest level number to put all three on a **0–1 scale**. Jev scores can be fractional, so keep the decimals.

This composite priority supplements the routing policy above; it does not replace its human-review check. The earlier `high`/`normal` label used frustration alone.

In [ ]:
# Define three scoring rubrics with different numbers of levels.
TRIAGE_QUESTIONS = {
    'severity': Score(
        instructions='How severe is the reported issue?',
        criteria=[
            'Cosmetic; no impact to functionality',
            'Broken or degraded feature, but workaround exists',
            'Blocking issue; no workaround exists',
        ],
    ),
    'frustration': Score(
        instructions='How frustrated is the customer?',
        criteria=[
            'Calm, just stating facts',
            'Frustrated but civil',
            'Very angry, strong language or threatening to leave',
        ],
    ),
    'report_quality': Score(
        instructions='How much does the report give an engineer to work with?',
        criteria=[
            'No detail; just says something is broken',
            'Names the feature but no steps or environment',
            'Steps to reproduce or environment, but not both',
            'Steps to reproduce and environment',
        ],
    ),
}

In [ ]:
# Show a detailed support ticket and ask all three Score questions together.
priority_state = {
    'ticket': (
        'On macOS 14 with Chrome 129, I open Billing, select October, and click Export. '
        'The page freezes every time and I cannot download the invoice. '
        'There is no other export option. This is extremely frustrating; '
        'I will cancel if this is not fixed.'
    ),
}
priority_record = ask_jev('composite-priority', priority_state, TRIAGE_QUESTIONS)

### Normalize before weighting

For example, a hypothetical severity score of 1.5 becomes `1.5 / 2 = 0.75`. A report-quality score of 2 becomes `2 / 3 ≈ 0.67`. These numbers illustrate the arithmetic; they are not API results.

`ask_jev` returns a recorded answer dictionary, so the helper reads `answer['score']`. Direct SDK answers expose the equivalent `.score` attribute. A missing answer remains unavailable and requires review.

In [ ]:
# Normalize a score using the highest level in that question's own rubric.
def normalized(answers, question_id):
    score = answers.get(question_id, {}).get('score')
    if score is None:
        return None
    top_level = len(TRIAGE_QUESTIONS[question_id].criteria) - 1
    return score / top_level

In [ ]:
# Define the business weights and combine only complete sets of normalized scores.
PRIORITY_WEIGHTS = {
    'severity': 0.6,
    'frustration': 0.3,
    'report_quality': 0.1,
}


def weighted_priority(answers):
    total = 0.0
    for question_id, weight in PRIORITY_WEIGHTS.items():
        score = normalized(answers, question_id)
        if score is None:
            return None
        total += weight * score
    return total

In [ ]:
# Display the raw scores, normalized values, and each contribution to priority.
if priority_record is not None:
    print('State analysed:')
    display(priority_state)
    component_rows = []
    for question_id, weight in PRIORITY_WEIGHTS.items():
        normalized_score = normalized(priority_record.answers, question_id)
        component_rows.append({
            'question': question_id,
            'raw_score': priority_record.answers.get(question_id, {}).get('score'),
            'top_level': len(TRIAGE_QUESTIONS[question_id].criteria) - 1,
            'normalized_score': normalized_score,
            'weight': weight,
            'contribution': None if normalized_score is None else weight * normalized_score,
        })
    display(pd.DataFrame(component_rows))
    priority_score = weighted_priority(priority_record.answers)
    if priority_score is None:
        print('Priority unavailable: a score is missing. Send this ticket to human review.')
    else:
        print(f'Weighted priority: {priority_score:.3f} (0–1 policy scale)')

### Interpret and change the policy

`priority = 0.6 × normalized severity + 0.3 × normalized frustration + 0.1 × normalized report quality`

The weights sum to 1, keeping the composite on a 0–1 scale. **This is not a probability or confidence score.** The weights express a business choice: severity matters most, and a detailed report makes investigation slightly easier. A vague report can still describe a serious issue; do not discard it because its report-quality score is low.

Try changing the weights while keeping them nonnegative and summing to 1. Rerun the calculation and display cells; you do not need another API call. For example, put more weight on severity and less on frustration. Does the priority change as expected? Validate the weights and any escalation thresholds against your own operational needs.

## Exercise

Imagine wrongly routing a billing complaint now costs much more than reviewing it. Change the development accuracy objective from 90% to 100%. Predict how coverage changes. Also explain one weakness that raising the confidence threshold cannot fix.

## Solution

Recompute on development cases only; keep held-out results as a final report. If no threshold qualifies, review all. Confidence alone cannot resolve missing categories, mixed intents or incorrect reference labels. A second independent mixed-intent judgment or deterministic multi-team rule may be needed; validate that revised workflow on new held-out cases.

In [ ]:
# Try a stricter accuracy target using only the development examples.
if development_results:
    stricter_threshold = choose_threshold(development_correct, development_confidence, target=1.0)
    print('Stricter threshold:', stricter_threshold if stricter_threshold is not None else 'review all')

Sources: [independent questions](https://docs.typesafe.ai/primitives), [confidence](https://docs.typesafe.ai/confidence). Thresholds and policy priorities in this notebook are teaching choices, not vendor recommendations.